In [ ]:
import sys
import pandas as pd
import numpy as np
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

sys.path.insert(0, ".")
from metrics import extract_features, FEATURE_NAMES

df = pd.read_csv("alz-speech.csv")
ling_feats = pd.DataFrame(df["transcript"].apply(extract_features).tolist())
df = pd.concat([df, ling_feats], axis=1)

X = df[["sex", "transcript"] + FEATURE_NAMES]
y = df["ad"]

preprocessor = ColumnTransformer(transformers=[
    ("tfidf", TfidfVectorizer(
        lowercase=True, stop_words="english",
        ngram_range=(1, 2), min_df=2, max_df=0.95
    ), "transcript"),
    ("ling", StandardScaler(), FEATURE_NAMES),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore")),
    ]), ["sex"]),
])

logreg_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(C=1.0, solver="liblinear", max_iter=5000, random_state=42)),
])

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
scoring = {"accuracy": "accuracy", "f1": "f1", "roc_auc": "roc_auc",
           "precision": "precision", "recall": "recall"}

results = cross_validate(logreg_pipeline, X, y, cv=cv, scoring=scoring, n_jobs=-1)

print("Logistic Regression (CV)")
for metric in scoring:
    s = results[f"test_{metric}"]
    print(f"  {metric:<10}: {s.mean():.4f}  (std {s.std():.4f})")

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

ages = df["age"].dropna().values

kde = gaussian_kde(ages, bw_method="scott")

n_samples = len(df)
synthetic_ages = kde.resample(n_samples).flatten()

# If ages should be integers or clipped to a plausible range
synthetic_ages = np.clip(np.round(synthetic_ages), ages.min(), ages.max()).astype(int)
x_grid = np.linspace(ages.min() - 10, ages.max() + 10, 500)

plt.figure(figsize=(8, 4))
plt.hist(ages, bins=20, density=True, alpha=0.5, label="Observed")
plt.plot(x_grid, kde(x_grid), label="KDE fit")
plt.xlabel("Age")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

sex_counts = df["sex"].value_counts()

plt.figure(figsize=(6, 4))
plt.bar(sex_counts.index, sex_counts.values, color=["steelblue", "salmon"], edgecolor="black")
plt.xlabel("Sex")
plt.ylabel("Count")
plt.title("Distribution of Sex")
for i, (label, count) in enumerate(sex_counts.items()):
    plt.text(i, count + 0.5, str(count), ha="center", va="bottom")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from metrics import extract_features, FEATURE_NAMES

df = pd.read_csv("alz-speech.csv")
ling_feats = pd.DataFrame(df["transcript"].apply(extract_features).tolist())
df = pd.concat([df, ling_feats], axis=1)

X = df[["sex", "transcript"] + FEATURE_NAMES]
y = df["ad"]

preprocessor = ColumnTransformer(transformers=[
    ("tfidf", TfidfVectorizer(
        lowercase=True, stop_words="english",
        ngram_range=(1, 2), min_df=2, max_df=0.95
    ), "transcript"),
    ("ling", StandardScaler(), FEATURE_NAMES),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore")),
    ]), ["sex"]),
])

svm_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", SVC(kernel="linear", C=1.0, probability=True, random_state=42)),
])

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
scoring = {"accuracy": "accuracy", "f1": "f1", "roc_auc": "roc_auc",
           "precision": "precision", "recall": "recall"}

results = cross_validate(svm_pipeline, X, y, cv=cv, scoring=scoring, n_jobs=-1)

print("Linear SVM (CV)")
for metric in scoring:
    s = results[f"test_{metric}"]
    print(f"  {metric:<10}: {s.mean():.4f}  (std {s.std():.4f})")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from metrics import extract_features, FEATURE_NAMES

df = pd.read_csv("alz-speech.csv")
ling_feats = pd.DataFrame(df["transcript"].apply(extract_features).tolist())
df = pd.concat([df, ling_feats], axis=1)

X = df[["sex", "transcript"] + FEATURE_NAMES]
y = df["ad"]

preprocessor = ColumnTransformer(transformers=[
    ("tfidf", TfidfVectorizer(
        lowercase=True, stop_words="english",
        ngram_range=(1, 2), min_df=2, max_df=0.95
    ), "transcript"),
    ("ling", StandardScaler(), FEATURE_NAMES),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore")),
    ]), ["sex"]),
])

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(n_estimators=200, max_depth=None, random_state=42, n_jobs=-1)),
])

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
scoring = {"accuracy": "accuracy", "f1": "f1", "roc_auc": "roc_auc",
           "precision": "precision", "recall": "recall"}

results = cross_validate(rf_pipeline, X, y, cv=cv, scoring=scoring, n_jobs=-1)

print("Random Forest (CV)")
for metric in scoring:
    s = results[f"test_{metric}"]
    print(f"  {metric:<10}: {s.mean():.4f}  (std {s.std():.4f})")

In [ ]:
import sys
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score
from xgboost import XGBClassifier

sys.path.insert(0, ".")
from metrics import extract_features, FEATURE_NAMES

def add_linguistic_features(df):
    feats = pd.DataFrame(df["transcript"].apply(extract_features).tolist())
    return pd.concat([df.reset_index(drop=True), feats], axis=1)

print("Extracting linguistic features for real data...")
real_df = add_linguistic_features(pd.read_csv("alz-speech.csv"))

print("Extracting linguistic features for paraphrased data...")
para_df = add_linguistic_features(pd.read_csv("alz-speech-paraphrased.csv"))

print(f"Done.  Real: {len(real_df)} rows  |  Paraphrased: {len(para_df)} rows")

LING_FEATURES = FEATURE_NAMES
FEATURES      = ["sex", "transcript"] + LING_FEATURES
TARGET        = "ad"
METRICS       = ["accuracy", "f1", "roc_auc", "precision", "recall"]

def fresh_models(seed):
    return {
        "LogReg": LogisticRegression(C=1.0, solver="liblinear", max_iter=5000, random_state=seed),
        "SVM":    SVC(kernel="linear", C=1.0, probability=True, random_state=seed),
        "RF":     RandomForestClassifier(n_estimators=200, random_state=seed, n_jobs=-1),
        "GBM":    GradientBoostingClassifier(n_estimators=200, random_state=seed),
        "XGB":    XGBClassifier(n_estimators=200, random_state=seed, eval_metric="logloss", verbosity=0),
    }

def build_preprocessor():
    return ColumnTransformer(transformers=[
        ("tfidf", TfidfVectorizer(
            lowercase=True, stop_words="english",
            ngram_range=(1, 2), min_df=2, max_df=0.95
        ), "transcript"),
        ("ling", StandardScaler(), LING_FEATURES),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("encode", OneHotEncoder(handle_unknown="ignore")),
        ]), ["sex"]),
    ])

def build_preprocessor_no_ling():
    return ColumnTransformer(transformers=[
        ("tfidf", TfidfVectorizer(
            lowercase=True, stop_words="english",
            ngram_range=(1, 2), min_df=2, max_df=0.95
        ), "transcript"),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("encode", OneHotEncoder(handle_unknown="ignore")),
        ]), ["sex"]),
    ])

def build_pipeline(model):
    return Pipeline([("preprocessor", build_preprocessor()), ("model", model)])

def build_pipeline_no_ling(model):
    return Pipeline([("preprocessor", build_preprocessor_no_ling()), ("model", model)])

def evaluate(pipeline, X_train, y_train, X_test, y_test):
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    return {
        "accuracy":  accuracy_score(y_test, y_pred),
        "f1":        f1_score(y_test, y_pred, zero_division=0),
        "roc_auc":   roc_auc_score(y_test, y_prob),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall":    recall_score(y_test, y_pred, zero_division=0),
    }

In [ ]:
from sklearn.model_selection import train_test_split

X_all = real_df[FEATURES]
y_all = real_df[TARGET]

scores_holdout = {name: {m: [] for m in METRICS} for name in fresh_models(0)}

for seed in range(100):
    X_train, X_test, y_train, y_test = train_test_split(
        X_all, y_all, test_size=0.2, stratify=y_all, random_state=seed
    )
    for name, model in fresh_models(seed).items():
        result = evaluate(build_pipeline(model), X_train, y_train, X_test, y_test)
        for metric, value in result.items():
            scores_holdout[name][metric].append(value)

print("80/20 Holdout — mean (std) over 100 random splits\n")
print(f"{'Metric':<12}", end="")
for name in scores_holdout:
    print(f"{name:>20}", end="")
print()
print("-" * (12 + 20 * len(scores_holdout)))
for metric in METRICS:
    print(f"{metric:<12}", end="")
    for name in scores_holdout:
        mean = np.mean(scores_holdout[name][metric])
        std  = np.std(scores_holdout[name][metric])
        print(f"  {mean:.4f} (std {std:.4f})", end="")
    print()

# Linguistic Features Ablation
# TF-IDF only vs TF-IDF + 15 linguistic features, across all models, 80/20 holdout over 20 seeds.

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

X_with    = real_df[FEATURES]
X_without = real_df[["sex", "transcript"]]
y_all     = real_df[TARGET]

ablation_scores = {
    name: {"with": {m: [] for m in METRICS}, "without": {m: [] for m in METRICS}}
    for name in fresh_models(0)
}

sss = StratifiedShuffleSplit(n_splits=100, test_size=0.2, random_state=0)
for seed, (idx_train, idx_test) in enumerate(sss.split(X_with, y_all)):
    y_true = y_all.iloc[idx_test]
    for name, model in fresh_models(seed).items():
        pipe_with = build_pipeline(model)
        pipe_with.fit(X_with.iloc[idx_train], y_all.iloc[idx_train])
        y_pred = pipe_with.predict(X_with.iloc[idx_test])
        y_prob = pipe_with.predict_proba(X_with.iloc[idx_test])[:, 1]
        ablation_scores[name]["with"]["accuracy"].append(accuracy_score(y_true, y_pred))
        ablation_scores[name]["with"]["f1"].append(f1_score(y_true, y_pred, zero_division=0))
        ablation_scores[name]["with"]["roc_auc"].append(roc_auc_score(y_true, y_prob))
        ablation_scores[name]["with"]["precision"].append(precision_score(y_true, y_pred, zero_division=0))
        ablation_scores[name]["with"]["recall"].append(recall_score(y_true, y_pred, zero_division=0))

        pipe_without = build_pipeline_no_ling(model)
        pipe_without.fit(X_without.iloc[idx_train], y_all.iloc[idx_train])
        y_pred = pipe_without.predict(X_without.iloc[idx_test])
        y_prob = pipe_without.predict_proba(X_without.iloc[idx_test])[:, 1]
        ablation_scores[name]["without"]["accuracy"].append(accuracy_score(y_true, y_pred))
        ablation_scores[name]["without"]["f1"].append(f1_score(y_true, y_pred, zero_division=0))
        ablation_scores[name]["without"]["roc_auc"].append(roc_auc_score(y_true, y_prob))
        ablation_scores[name]["without"]["precision"].append(precision_score(y_true, y_pred, zero_division=0))
        ablation_scores[name]["without"]["recall"].append(recall_score(y_true, y_pred, zero_division=0))

print("Linguistic features ablation — 100 splits\n")
for name in ablation_scores:
    print(f"\n{name}")
    print("=" * 56)
    print(f"{'Metric':<12}  {'Without ling':>14}  {'With ling':>14}  {'Delta':>8}")
    print("-" * 56)
    for metric in METRICS:
        wo    = np.mean(ablation_scores[name]["without"][metric])
        wi    = np.mean(ablation_scores[name]["with"][metric])
        delta = wi - wo
        sign  = "+" if delta >= 0 else ""
        print(f"  {metric:<10}  {wo:.4f}          {wi:.4f}      {sign}{delta:.4f}")

In [ ]:
CONFIGS = {
    "Real Only":         {"use_real": True,  "use_para": False},
    "Real + Paraphrase": {"use_real": True,  "use_para": True},
    "Paraphrase Only":   {"use_real": False, "use_para": True},
}

VARIANTS = {
    "with_ling":    {"features": FEATURES,              "builder": build_pipeline},
    "without_ling": {"features": ["sex", "transcript"], "builder": build_pipeline_no_ling},
}

MODEL_NAMES = list(fresh_models(0).keys())

ad_subjects   = real_df[real_df["ad"] == 1]["subject"].unique().tolist()
ctrl_subjects = real_df[real_df["ad"] == 0]["subject"].unique().tolist()
n_test_ad     = max(1, int(len(ad_subjects)   * 0.20))
n_test_ctrl   = max(1, int(len(ctrl_subjects) * 0.20))

scores = {
    var: {cfg: {m: {met: [] for met in METRICS} for m in MODEL_NAMES} for cfg in CONFIGS}
    for var in VARIANTS
}

for seed in range(100):
    rng = np.random.default_rng(seed)

    test_subjects  = set(rng.choice(ad_subjects,   n_test_ad,   replace=False).tolist()
                       + rng.choice(ctrl_subjects, n_test_ctrl, replace=False).tolist())
    train_subjects = set(real_df["subject"].unique()) - test_subjects

    for var_name, var in VARIANTS.items():
        feat_cols = var["features"]
        builder   = var["builder"]

        test_data = real_df[real_df["subject"].isin(test_subjects)]
        X_test = test_data[feat_cols]
        y_test = test_data[TARGET].astype(int)

        for cfg_name, cfg in CONFIGS.items():
            train_parts = []
            if cfg["use_real"]:
                train_parts.append(real_df[real_df["subject"].isin(train_subjects)])
            if cfg["use_para"]:
                train_parts.append(para_df[para_df["subject"].isin(train_subjects)])

            train_data = pd.concat(train_parts, ignore_index=True)
            X_train = train_data[feat_cols]
            y_train = train_data[TARGET].astype(int)

            for model_name, model in fresh_models(seed).items():
                result = evaluate(builder(model), X_train, y_train, X_test, y_test)
                for metric, value in result.items():
                    scores[var_name][cfg_name][model_name][metric].append(value)

print("Done. Run the next cell to see results.")

In [ ]:
for model_name in MODEL_NAMES:
    for var_name in VARIANTS:
        label = "With ling features" if var_name == "with_ling" else "Without ling features"
        print(f"\n{model_name}  —  {label}")
        print("=" * 72)
        print(f"{'Metric':<12}", end="")
        for cfg_name in CONFIGS:
            print(f"{cfg_name:>22}", end="")
        print()
        print("-" * 72)
        for metric in METRICS:
            print(f"{metric:<12}", end="")
            for cfg_name in CONFIGS:
                vals = scores[var_name][cfg_name][model_name][metric]
                mean, std = np.mean(vals), np.std(vals)
                print(f"  {mean:.3f} (std {std:.3f})   ", end="")
            print()

In [ ]:
import scipy.stats as stats

# Toggle this to switch what the recall delta measures
USE_LING_FEATURES = False   # True = use linguistic features; False = TF-IDF only

var_key = "with_ling" if USE_LING_FEATURES else "without_ling"
feat_label = "with ling features" if USE_LING_FEATURES else "without ling features"

print(f"Recall delta: Real + Paraphrase vs Real Only  ({feat_label}, 100 seeds)\n")
print(f"{'Model':<10}  {'Mean':>8}  {'Std':>8}  {'Median':>8}  {'% > 0':>8}  {'p-value':>10}")
print("-" * 58)

all_deltas = []
for model_name in MODEL_NAMES:
    para = np.array(scores[var_key]["Real + Paraphrase"][model_name]["recall"])
    base = np.array(scores[var_key]["Real Only"][model_name]["recall"])
    d    = para - base
    all_deltas.extend(d.tolist())
    _, p = stats.ttest_1samp(d, 0)
    print(f"  {model_name:<10}  {d.mean():>+8.4f}  {d.std():>8.4f}  "
          f"{np.median(d):>+8.4f}  {100*np.mean(d>0):>7.1f}%  {p:>10.4f}")

d_all = np.array(all_deltas)
_, p = stats.ttest_1samp(d_all, 0)
print("-" * 58)
print(f"  {'ALL':<10}  {d_all.mean():>+8.4f}  {d_all.std():>8.4f}  "
      f"{np.median(d_all):>+8.4f}  {100*np.mean(d_all>0):>7.1f}%  {p:>10.4f}")
print("\n* p-value: one-sample t-test, H0: paraphrasing has no effect on recall")